<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week8/week8_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
import torchaudio.transforms as AT
import torch.nn as nn
import csv
import joblib
import os

# set seed
torch.manual_seed(801)

# load data

if not os.path.exists('SMART-2026'):
    !git clone https://github.com/801-Hillside-Terrace/SMART-2026.git

data = np.load("SMART-2026/project/data/GTZAN_processed/gtzan_processed.npz")

X_train = data["X_train"]
X_val = data["X_val"]
X_test = data["X_test"]

y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

print(X_train.shape)
print(X_train[0].shape)

# convert to tensors and normalize (divide by 255)

X_train = torch.tensor(X_train, dtype=torch.float32) / 255
X_val   = torch.tensor(X_val, dtype=torch.float32) / 255
X_test  = torch.tensor(X_test, dtype=torch.float32) / 255

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

# arrange in correct order for input:

X_train = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
X_val   = torch.tensor(X_val, dtype=torch.float32).permute(0, 3, 1, 2)
X_test  = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)

# standardize
mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std
X_test  = (X_test - mean) / std

# transformation for training (randomly erase part of the image)
train_transform = T.Compose([T.RandomApply([T.RandomErasing(p=1.0,scale=(0.02, 0.15),ratio=(0.3, 3.3))], p=0.5)])

freq_mask = AT.FrequencyMasking(freq_mask_param=20)
time_mask = AT.TimeMasking(time_mask_param=30)

# dataloaders:

batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

(699, 128, 128, 3)
(128, 128, 3)


/tmp/ipykernel_2874/1049498562.py:49: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
/tmp/ipykernel_2874/1049498562.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val   = torch.tensor(X_val, dtype=torch.float32).permute(0, 3, 1, 2)
/tmp/ipykernel_2874/1049498562.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test  = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)


So far, we have looked at tabular data, where the observations are rows and the features are columns.  Tabular data can still have patterns that involve interactions between the features, but we do not initially assume any sort of structure like this.  

Now imagine that our features are no longer tabular, but instead images.  Images innately have some kind of structure, such as nearby pixels are usually related to eachother as together they form different parts of the image.  Even a relatively small 100x100x1 pixel grayscale image would require an input dimension of 10,000 and even though we may be able to fit this, the rest of the network likely needs a lot of complexity to correctly learn good representations of the images and an enormous amount of data.

A solution is then in the form of trying to exploit the known structure within images, there are roughly three key ideas:

1.  Translation invariance: The network should be able to recognize the same feature regardless of where it appears within the image.  (Useful features can appear anywhere in the image, they don't necessarily consistently appear in the top left etc.)

2.  Locality: The early layers in the network should focus on smaller local regions of the image rather than focus on the entire image at once. (Pixels near eachother are more related than those further away)

3.  Hierarchy: Deeper layers combine the local patterns into complex and long-range features.  (We can construct higher-level vision from the lower-level features, complex objects can be built from simpler pieces)

Let's use Where's Waldo as an example:

1.  Waldo can appear anywhere in an image.

2.  We examine specific regions of the image rather than looking at the image as a whole to find him.

3.  Waldo is made of edges and colors, which when put together may be stripes or glasses, and ultimately composed into Waldo.

Let's imagine we have an image with a lot in it and want to search for Waldo.  One way to do so that is aligned with the previous 3 points is to use a small pattern detector that sweeps across the image looking for certain local patterns.  

Remember that an image is just a grid of pixels meaning we can represent it as a tensor where the third dimension is the amount of channels (RGB images have three channels, grayscale have one).   

In practice the pattern detector spoken about is called a kernel, and it is just a grid of numbers (where the numbers are learnable weights).  Here is an example for a single channel image with the math made explicit:

<img src='https://d2l.ai/_images/correlation.svg' width = 400>

$$\begin{split}0\times0+1\times1+3\times2+4\times3=19,\\
1\times0+2\times1+4\times2+5\times3=25,\\
3\times0+4\times1+6\times2+7\times3=37,\\
4\times0+5\times1+7\times2+8\times3=43.\end{split}$$

Everything is multiplied element-wise and then summed together (and in practice there is also an added bias term not shown here).  The sum is then output as an element in a new channel, and the kernel then moves to the next location and repeats until it has swept across the entire image input and created a new channel.  In practice, since inputs are often multi-channel, kernels are as well (if this input had 3 channels, the kernel would also have 3 channels and would therefore have 12 weights, 4 for each channel, but would still only output the 2x2).  

We often run multiple kernels across the same input, creating several new channels, and then those output channels will be plugged into an activation function (typically ReLU) and have the next layer of kernels take those as inputs making the network deeper.  Each of these layers is referred to as a convolution layer (hence the name).  There are also typically normalizations thrown in too like a 2-dimensional batchnorm (computes a mean and variance of all pixels on each channel), and after the activation function is applied "pooling" is often performed per channel which reduces the size of the outputs, for example a 2x2 max pooling window would reduce the 2x2 output in the above example to just the 1x1 number 43 (and there is also average pooling, etc).  After all the convolution layers, modern networks perform some sort of final pooling to aggregate everything for plugging into a fully connected neural network to perform the final task (typically classification or regression).  For example, if the final convolution layer output 512 channels of size 7x7, we could take the average of each of those 7x7 channels and input the 512 averages into the fully connected network.  

Below, we take an existing convolutional neural network architecture (ResNet18) and load pre-existing weights trained for a fairly different task (classifying images with 1000 possible classes).  We then "freeze" all the layers but the final convolution layer and the fully connected/output layer to re-train for our specific task of using the mel spectrograms to predict music genre.  This practice is called "transfer learning" and typically works as the early layers are more general while the later layers are what require fine-tuning for specific tasks.  This speeds up computation time/tuning time etc. and is often done in practice.  

In [ ]:
# load model
model = models.resnet18(weights="IMAGENET1K_V1") # load weights

# replace final layer
model.fc = nn.Linear(model.fc.in_features, 10)  # 10 genres

# set device up
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

# freeze all layers
for param in model.parameters():
    param.requires_grad = False
# unfreeze layer 4 to train it (last convolution layer)
for param in model.layer4.parameters():
    param.requires_grad = True
# unfreeze final layer to train it (fully connected/output layer)
for param in model.fc.parameters():
    param.requires_grad = True

# loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

In [ ]:
# training:

num_epochs = 100

# empty lists to store
train_losses = []
val_losses = []

train_accs = []
val_accs = []

for epoch in range(num_epochs):
  # training loop
  model.train()
  running_loss = 0.0
  running_correct = 0
  total = 0

  for xb, yb in train_loader: # loop through batches in training loader
      xb = xb.to(device)
      yb = yb.to(device)

      # apply transformations
      xb = train_transform(xb)
      #xb = freq_mask(xb)
      #xb = time_mask(xb)

      # forward pass
      preds = model(xb)

      loss = criterion(preds, yb)

      # backward pass
      optimizer.zero_grad()

      loss.backward()

      optimizer.step()

      # track loss and accuracy
      running_loss += loss.item() * xb.size(0)

      predicted_classes = preds.argmax(dim=1)

      running_correct += (predicted_classes == yb).sum().item()

      total += yb.size(0)

  train_loss = running_loss / total
  train_acc = running_correct / total

  train_losses.append(train_loss)
  train_accs.append(train_acc)

  # validation loop
  model.eval()

  val_running_loss = 0.0
  val_running_correct = 0
  val_total = 0

  with torch.no_grad():

      for xb, yb in val_loader:

          xb = xb.to(device)
          yb = yb.to(device)

          preds = model(xb)

          loss = criterion(preds, yb)

          val_running_loss += loss.item() * xb.size(0)

          predicted_classes = preds.argmax(dim=1)

          val_running_correct += (predicted_classes == yb).sum().item()

          val_total += yb.size(0)

  val_loss = val_running_loss / val_total
  val_acc = val_running_correct / val_total

  val_losses.append(val_loss)
  val_accs.append(val_acc)

  print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

In [ ]:
# evaluation on test data:

model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        preds = model(xb)

        predicted_classes = preds.argmax(dim=1)

        test_correct += (predicted_classes == yb).sum().item()

        test_total += yb.size(0)

test_acc = test_correct / test_total

print(f'Test Accuracy: {test_acc:.4f}')

